# Exploratory clean Task 2 retention study

This bounded follow-up reuses the verified five-class Task 1 best checkpoint, fixed disk-backed splits, and frozen continual preprocessor. It does **not** retrain Task 1 and does not run locally. Two clean Task 2 configurations share the same architecture, expanded initial weights, seed, batch size, and 70,576 draws per epoch. Historical Task 2 test results have already been inspected, so this is explicitly exploratory rather than a fresh confirmatory test.

## Predeclared hypotheses and comparison

These are hypotheses, **not established causes**:

- Uniform sampling over eight supplied classes gives Benign 12.5% expected exposure per epoch.
- The existing 500-row replay contains only 100 distinct examples per old class.
- The historical Task 2 run inherited learning rate 0.001.

Configuration A keeps the existing 500-row replay and current uniform sampler, changing only learning rate to 0.0001. Configuration B uses learning rate 0.0001 plus 5,000 distinct training-only replay rows per old class; exactly 62.5% of draws go to old classes in their original Task 1 training proportions and 37.5% go uniformly to the three new classes. B is a **combined replay-size and sampling intervention** and cannot isolate either effect.

Before execution, the selection rule is fixed: maximize full clean-validation macro-F1. Exact ties use lower validation Benign FPR, then higher old-class focused macro-F1, then the stable configuration name. Benign FPR, old retention, and new-class performance are always reported. Test data cannot select a checkpoint or configuration.

## 1. Pin the published source and verify Kaggle

Enable a Kaggle GPU and Internet. This checkout is detached at the source commit published before this notebook was created. Kaggle's CUDA-enabled PyTorch is retained. Focused sampler, split-isolation, durable checkpoint/resume, and continual-metric tests must pass without skips.

In [ ]:
from pathlib import Path
import os, sys, subprocess, re, shutil, json, platform
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '57009efe09f48ee742576bcc5d089a71c30b4f0e'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-task2-retention')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import torch
import matplotlib
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required: enable a Kaggle GPU and restart the session')
print({'source_sha': checked_out_sha, 'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'gpu_count': torch.cuda.device_count()})
subprocess.run([sys.executable, '-m', 'py_compile', 'scripts/run_task2_clean_retention.py'], cwd=REPO_DIR, check=True)
for pattern in ('test_task2_replay.py', 'test_disk_replay.py', 'test_full_disk_training.py', 'test_full_continual_metrics.py'):
    result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern, '-v'], cwd=REPO_DIR, check=True, capture_output=True, text=True)
    print(result.stdout, end=''); print(result.stderr, end='')
    summary = result.stdout + result.stderr
    count = re.search(r'Ran (\d+) tests? in', summary)
    if count is None or int(count.group(1)) == 0 or re.search(r'OK \(skipped=\d+\)', summary):
        raise RuntimeError(f'Focused tests absent or skipped: {pattern}')

## 2. Select the two read-only Kaggle inputs

Attach (1) the complete prepared-data folder `review2_large_prepare_<id>` and (2) the complete verified Task 1 folder `full_task1_20261003T074354_763068Z` as private Kaggle inputs. The runner requires all prepared `.npy` arrays/manifests plus the Task 1 `best.pt`, manifests, continual preprocessor, replay buffer, and Task 1-before test metrics. It verifies the prepared source/input hashes, Task 1 checkpoint hash, fixed split counts, preprocessing scope/hash, replay dimensions, training-only row provenance, and single-pass transformation before training.

In [ ]:
PREPARED_DIR = None  # e.g. '/kaggle/input/review2-prepared/review2_large_prepare_20261003T...'
TASK1_RUN_DIR = None  # e.g. '/kaggle/input/review2-task1/full_task1_20261003T074354_763068Z'
OUTPUT_DIR = Path('/kaggle/working/task2_clean_retention_study')
RESUME_RUN_DIR = None  # optional complete prior output under /kaggle/input or /kaggle/working
MAX_EPOCHS_THIS_CALL = None  # optional positive epoch-boundary pause for each active arm
def choose_folder(explicit, prefix, marker):
    if explicit is not None:
        folder = Path(explicit)
    else:
        candidates = [p.parent for base in (Path('/kaggle/input'), Path('/kaggle/working')) for p in base.rglob(marker) if prefix in p.parent.name]
        print(prefix, 'candidates:', [str(path) for path in candidates])
        if len(candidates) != 1:
            raise RuntimeError(f'Found {len(candidates)} {prefix} folders; set the selector explicitly')
        folder = candidates[0]
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    return folder
prepared_dir = choose_folder(PREPARED_DIR, 'review2_large_prepare_', 'features_manifest.json')
task1_dir = choose_folder(TASK1_RUN_DIR, 'full_task1_', 'final_manifest.json')
if RESUME_RUN_DIR is not None:
    previous = Path(RESUME_RUN_DIR)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    if previous.resolve().is_relative_to(Path('/kaggle/working')):
        OUTPUT_DIR = previous
    else:
        if OUTPUT_DIR.exists():
            raise FileExistsError(f'Refusing to overwrite {OUTPUT_DIR}')
        shutil.copytree(previous, OUTPUT_DIR)
print({'prepared_dir': str(prepared_dir), 'task1_dir': str(task1_dir), 'output_dir': str(OUTPUT_DIR)})

## 3. Run or resume the bounded two-configuration study

Each arm starts by independently loading the same Task 1 best checkpoint and expanding with seed 42; the runner rejects unequal initial state hashes. AdamW is recreated at learning rate 0.0001. Both arms use batch size 256, exactly 70,576 draws (276 steps) per epoch, at most eight epochs, and patience three on full clean-validation macro-F1. `latest.pt` is authoritative at epoch boundaries and `best.pt` is macro-F1 selected. Actual per-class draws and unique rows drawn per class are retained in every epoch record. Replay features are already transformed and are never transformed again.

After both checkpoints and their clean-validation reports exist, `selection_lock.json` is written. Only then is the selected checkpoint evaluated once across the fixed old/new test rows. Unselected Task 2 checkpoints are never opened on test data.

In [ ]:
command = [sys.executable, str(REPO_DIR / 'scripts/run_task2_clean_retention.py'), '--prepared-dir', str(prepared_dir), '--task1-run-dir', str(task1_dir), '--output-dir', str(OUTPUT_DIR)]
if MAX_EPOCHS_THIS_CALL is not None:
    command.extend(['--max-epochs-this-call', str(MAX_EPOCHS_THIS_CALL)])
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)

## 4. Review and preserve the result

The final display reads saved JSON only. It reports whether the selected fixed-test combined accuracy reaches 85%, but the saved per-class recall/F1, old/new metrics, Benign FPR, forgetting, and normalized confusion plots remain first-class evidence. Success is not guaranteed. Preserve the entire output folder as a private Kaggle dataset/version; do not commit checkpoints, prepared arrays, or generated results. Historical Task 2 result archives remain separate and unchanged.

In [ ]:
study = json.loads((OUTPUT_DIR / 'study_manifest.json').read_text())
selection = json.loads((OUTPUT_DIR / 'selection_lock.json').read_text())
result = json.loads((OUTPUT_DIR / 'selected_test_summary.json').read_text())
assert study['source_sha'] == selection['source_sha'] == SOURCE_SHA
assert selection['test_partition_opened_before_lock'] is False and result['test_used_for_selection'] is False
print('Selected configuration:', selection['selected_condition'])
print('Combined test accuracy:', result['combined_test']['accuracy'])
print('Combined test macro-F1:', result['combined_test']['macro_f1'])
print('Benign FPR:', result['benign_fpr'])
print('Old/new focused macro-F1:', result['old_focused_macro_f1'], result['new_focused_macro_f1'])
print('Forgetting:', result['forgetting'])
print('85% combined-accuracy target reached:', result['target_reached'])
print('Per-class recall/F1 (do not hide minority failures):', json.dumps(result['per_class_recall_f1'], indent=2))
print('Save the complete private output folder:', OUTPUT_DIR)